# Fase 2 — Claude Sonnet 5.5 / Haiku 5.5 · SOLO OFFLINE

Questo notebook NON esegue alcuna inferenza Anthropic e NON modifica i dati V3 o Gemini 5+1. Usa una directory Colab separata da `/content/Tesi_OpenRCA`.

Prima dei test API: verifica la versione codice, le dipendenze, la suite Python e la disponibilità del Secret `ANTHROPIC_API_KEY` (non lo stampa). **Non modificare la vecchia sessione Colab congelata per Gemini.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

CLAUDE_REPO = Path('/content/Tesi_OpenRCA_Claude')
CLAUDE_DATA = Path('/content/drive/MyDrive/Tesi_OpenRCA')
BRANCH = 'feature/claude-cross-model-2026-11-05'
URL = 'https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git'

if not CLAUDE_REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,
                    URL, str(CLAUDE_REPO)], check=True)

def claude_git(*args):
    return subprocess.run(['git', '-C', str(CLAUDE_REPO), *args],
                          check=True, text=True,
                          capture_output=True).stdout.strip()

assert claude_git('branch', '--show-current') == BRANCH
assert not claude_git('status', '--porcelain')
print('Branch Claude:', BRANCH)
print('Commit Claude:', claude_git('rev-parse', 'HEAD'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(CLAUDE_REPO / 'requirements.txt')], check=True)
try:
    print('LangChain Anthropic:', version('langchain-anthropic'))
except PackageNotFoundError as exc:
    raise RuntimeError('Manca langchain-anthropic') from exc

for rel in ['data/processed/split_market_v1.csv',
            'data/processed/benchmark_inputs_market.csv', 'Market.zip']:
    p = CLAUDE_DATA / rel
    print(rel, 'PRESENTE' if p.exists() else 'ASSENTE')

print('Bootstrap Claude OFFLINE completato: nessuna API chiamata.')

## Test di regressione senza chiave API

Prima si eseguono i test del provider Claude, poi la suite generale. Nessuna inferenza a pagamento: per i test del provider si usano mock.

In [ ]:
env = {**os.environ, 'PYTHONPATH': str(CLAUDE_REPO / 'src')}
for mask in ['test_anthropic_model_factory.py', 'test_*.py']:
    print('TEST:', mask)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                    '-s', str(CLAUDE_REPO / 'src/tests'),
                    '-p', mask, '-v'],
                   cwd=CLAUDE_REPO, env=env, check=True)
print('Suite terminata. Nessuna API Claude chiamata.')

## Verifica del Secret — nessun costo

Apri Colab → pannello **Secrets** (icona chiave) → aggiungi **ANTHROPIC_API_KEY** → incolla il valore consegnato dal professore → abilita l'accesso al notebook. Non inserirlo qui come testo e non salvarlo su Drive o GitHub.

La seguente cella legge solo il Secret. Una chiave presente non conferma che sia valida o che l'account disponga di credito; quello richiede uno smoke test successivo e autorizzato.

In [ ]:
from google.colab import userdata
secret = userdata.get('ANTHROPIC_API_KEY')
assert isinstance(secret, str) and secret.strip(), (
    'Secret ANTHROPIC_API_KEY assente o non accessibile.'
)
os.environ['ANTHROPIC_API_KEY'] = secret
del secret

print('ANTHROPIC_API_KEY accessibile: SI')
print('Modelli in piano: claude-sonnet-5-5, claude-haiku-5-5')
print('Nessuna inferenza, nessuna chiamata API, nessun file modificato.')


## Piano Sonnet 40 + Haiku 40, preview GT-free e costo indicativo

Questa cella usa solo il piano V3 e i **contatori dei token storici**. Non legge `query.csv`, non legge le predizioni né punteggi, non avvia API e non scrive file. Calcola un costo di riferimento per 40 run Sonnet, **non un limite garantito**. Prima del batch servono test verdi, una prova reale autorizzata e un budget deciso esplicitamente.

In [ ]:
import json, sys
if str(CLAUDE_REPO / 'src') not in sys.path:
    sys.path.insert(0, str(CLAUDE_REPO / 'src'))
from evaluation import claude_development_campaign as cdc

piano_preview = cdc.build_plan(CLAUDE_DATA, claude_git('rev-parse', 'HEAD'))
assert piano_preview['run_count'] == 80
assert [r['model_key'] for r in piano_preview['runs']] == (
    ['sonnet'] * 40 + ['haiku'] * 40
)
print('Piano:', piano_preview['protocol'])
print('Commit:', piano_preview['git_commit'])
print('Varianti:', ', '.join(piano_preview['variants']))
print('Incidenti:', piano_preview['case_count'])
print('Run Sonnet:', sum(r['model_key']=='sonnet' for r in piano_preview['runs']))
print('Run Haiku:', sum(r['model_key']=='haiku' for r in piano_preview['runs']))

source_path = (CLAUDE_DATA / 'data/protocols/v2'
               / 'openrca_harness_v2_five_agents_development_v3_plan_v1.json')
source = json.loads(source_path.read_text(encoding='utf-8'))
old_results = CLAUDE_DATA / 'data/results/v2' / source['protocol']
total_in, total_out, complete_usage, found = 0, 0, 0, 0
for case in source['cases']:
    for run_id in case['run_ids']:
        fp = old_results / (run_id + '.json')
        if not fp.exists():
            continue
        found += 1
        metrics = json.loads(fp.read_text(encoding='utf-8')).get('metrics', {})
        if (metrics.get('token_usage_complete') is True and
            type(metrics.get('input_tokens')) is int and
            type(metrics.get('output_tokens')) is int):
            complete_usage += 1
            total_in += metrics['input_tokens']
            total_out += metrics['output_tokens']
print('Storico V3 leggibile:', found, '/ 40')
print('Storico con token completi:', complete_usage, '/ 40')
print('Token input su run complete:', total_in)
print('Token output su run complete:', total_out)
if complete_usage:
    sonnet_estimate = (total_in * 2 + total_out * 10) / 1_000_000
    sonnet_estimate *= 40 / complete_usage
    print('Sonnet 40 run, costo solo indicativo a token V3: '
          f'USD {sonnet_estimate:.3f}')
    print('Se Claude usa il doppio dei token: '
          f'USD {2*sonnet_estimate:.3f}')
    print('Se Claude usa quattro volte i token: '
          f'USD {4*sonnet_estimate:.3f}')
print('PREVIEW SOLO OFFLINE - zero API, zero scoring, zero scritture.')


## Congelamento create-only del piano Claude

**Verifica i 330+ test della vecchia campagna e, soprattutto, l'esito dei nuovi test del branch Claude appena eseguiti in questo notebook.** Poi controlla l'elenco delle 5 varianti e la stima. Metti `CREA_PIANO_CLAUDE=True` solo quando la preview è corretta. Questa cella non chiama Claude. Non lanciare alcuna campagna a pagamento finché non avremo confermato lo smoke test e un budget esplicito.

In [ ]:
CREA_PIANO_CLAUDE = False  # True SOLO dopo preview e test offline

p = cdc._plan_path(CLAUDE_DATA)
if CREA_PIANO_CLAUDE:
    if p.exists():
        print('Piano già presente: non sovrascrivo.')
    else:
        print('Creato piano:', cdc.write_plan(CLAUDE_DATA, CLAUDE_REPO))
else:
    print('Nessuna scrittura su Drive.')

if p.exists():
    state = cdc.preflight(CLAUDE_DATA, CLAUDE_REPO)
    print('Stato totale:', state['counts'])
    print('Sonnet:', state['models']['sonnet'])
    print('Haiku:', state['models']['haiku'])
    print('SHA piano:', state['plan_sha256'])
else:
    print('Piano non ancora creato: solo preview.')

print('Nessuna inferenza Anthropic chiamata.')


## Launcher 40 Sonnet — DISATTIVATO per impostazione predefinita

La seguente cella è disponibile **solo dopo** aver verificato tutte le suite offline, il piano create-only su Drive, la compatibilità di tool calling con una prima run pilota e un budget esplicitamente approvato. Rischio: `BUDGET_RIFERIMENTO_USD` **non è un limite rigido di spesa Anthropic**. Massimo 8 nuove run per cella, sempre sequenziali, JSON+SHA persistente per ciascuna. Le run fallite vengono conservate, mai ripetute automaticamente.

**Non eseguire a pagamento finché non abbiamo analizzato la preview e approvato la spesa.**

In [ ]:
from evaluation.claude_development_campaign import run_batch

ESEGUI_SONNET = False  # NON modificare prima dell'approvazione del budget
BUDGET_RIFERIMENTO_USD = 0.0  # Da fissare prima di qualsiasi API
MAX_NUOVE_RUN = 1  # Prima SOLO una prova, successivamente 1..8

if not ESEGUI_SONNET:
    print('API Claude DISABILITATE. Nessuna run avviata.')
else:
    assert BUDGET_RIFERIMENTO_USD > 0
    assert 1 <= MAX_NUOVE_RUN <= 8
    risultato_batch = run_batch(
        CLAUDE_DATA, CLAUDE_REPO,
        model_key='sonnet',
        max_new_runs=MAX_NUOVE_RUN,
        confirm_api=True,
        approved_budget_usd=BUDGET_RIFERIMENTO_USD,
    )
    print('Nuove run completate:', risultato_batch['new_run_count'])
    print('Stato Sonnet:', risultato_batch['counts'])


## Dopo le 40 run: freeze Sonnet, poi scoring offline

Confronta prima lo stato di tutte le 40 run. Il freeze è consentito soltanto quando non ci sono `ready`, `CLAIM_PENDING` o `REVIEW_INTEGRITY`. Gli eventuali errori provider già registrati non vengono cancellati. La creazione del freeze è disabilitata per impostazione predefinita e non consulta query.csv né alcun file di ground truth.

In [ ]:
from evaluation import claude_development_freeze as cdf

CREA_FREEZE_SONNET = False  # SOLO quando i 40 tentativi sono stati verificati
stato_campagna = cdc.preflight(CLAUDE_DATA, CLAUDE_REPO)
print('Sonnet:', stato_campagna['models']['sonnet'])
print('Haiku:', stato_campagna['models']['haiku'])

if CREA_FREEZE_SONNET:
    freeze = cdf.build_freeze(CLAUDE_DATA, CLAUDE_REPO, 'sonnet')
    path = cdf.freeze_path(CLAUDE_DATA, 'sonnet')
    cdc._write_only_new(path, freeze)
    print('Freeze Sonnet create-only:', path)
    print('Freeze SHA-256:', cdc._sha(path))
else:
    print('Freeze non creato; scoring non avviato.')
